# Wiener-Hunt Regularized Image Deconvolution
### Computational Image Processing & Inverse Problems
**Author**: Mohammed EL BARAKA  
**Topic**: Linear Inverse Problems, Fourier Diagonalization, Tikhonov Regularization, and Bias-Variance Tradeoff.

## 1. Setup and Package Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add python directory to path
sys.path.append(str(Path.cwd().parent / "python"))

from deconvolution import (
    load_dataset,
    wiener_deconvolve,
    compute_otf,
    compute_all_metrics,
    apply_edge_taper,
    RegularizationType
)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Load Datasets
- **Data1**: Degraded by Gaussian-like blur and additive noise (smooth isotropic decay, (
u) > 0$).
- **Data2**: Degraded by  	imes 7$ Box / Motion blur (anisotropic with exact zero-crossings).

In [ ]:
d1 = load_dataset(1)
d2 = load_dataset(2)
truth = d1["ground_truth"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(truth, cmap="gray")
axes[0].set_title("Ground Truth ^*$", fontweight="bold")
axes[1].imshow(d1["blurred"], cmap="gray")
axes[1].set_title("Data1 (Gaussian Blur)", fontweight="bold")
axes[2].imshow(d2["blurred"], cmap="gray")
axes[2].set_title("Data2 (Box Blur)", fontweight="bold")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Optical Transfer Function (OTF) Analysis
We analyze the Fourier transform of the PSF kernels to inspect their frequency attenuation and zero-crossings.

In [ ]:
H1 = compute_otf(d1["psf"], (256, 256), center_psf=True)
H2 = compute_otf(d2["psf"], (256, 256), center_psf=True)

freq_x = np.fft.fftshift(np.fft.fftfreq(256))

plt.figure(figsize=(10, 4))
plt.plot(freq_x, np.abs(np.fft.fftshift(H1))[128, :], "b-", lw=2, label="Gaussian PSF ($|H_1| > 0$)")
plt.plot(freq_x, np.abs(np.fft.fftshift(H2))[128, :], "r--", lw=2, label="Box PSF ($|H_2|$ zero crossings)")
plt.axhline(0, color="gray", linestyle=":")
plt.xlabel("Normalized Spatial Frequency $
u_x$")
plt.ylabel("Magnitude $|H(
u_x, 0)|$")
plt.title("Optical Transfer Function Frequency Slices", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()

## 4. Wiener-Hunt Inversion and Lambda Optimization
We solve the regularized problem:
63625\hat{X}(
u) = rac{H^*(
u)}{|H(
u)|^2 + \lambda |D(
u)|^2} Y(
u)63625

In [ ]:
lambdas = np.logspace(-6, 2, 80)
errs1, errs2 = [], []

for lam in lambdas:
    r1, _ = wiener_deconvolve(d1["blurred"], d1["psf"], reg_param=lam, reg_type="gradient")
    r2, _ = wiener_deconvolve(d2["blurred"], d2["psf"], reg_param=lam, reg_type="gradient")
    errs1.append(compute_all_metrics(r1, truth)["relative_l2"])
    errs2.append(compute_all_metrics(r2, truth)["relative_l2"])

opt_lam1 = lambdas[np.argmin(errs1)]
opt_lam2 = lambdas[np.argmin(errs2)]

plt.figure(figsize=(10, 4.5))
plt.semilogx(lambdas, errs1, "b-", lw=2, label=f"Data1 (Opt $\lambda={opt_lam1:.2e}$, Err={min(errs1):.4f})")
plt.semilogx(lambdas, errs2, "m-", lw=2, label=f"Data2 (Opt $\lambda={opt_lam2:.2e}$, Err={min(errs2):.4f})")
plt.xlabel("Regularization Parameter $\lambda$")
plt.ylabel("Relative L2 Error $\Delta_2$")
plt.title("Convex U-Curve Optimization", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.show()

## 5. Visual Comparison of Optimal Restorations

In [ ]:
best1, _ = wiener_deconvolve(d1["blurred"], d1["psf"], reg_param=opt_lam1, reg_type="gradient")
best2, _ = wiener_deconvolve(d2["blurred"], d2["psf"], reg_param=opt_lam2, reg_type="gradient")

m1 = compute_all_metrics(best1, truth)
m2 = compute_all_metrics(best2, truth)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(truth, cmap="gray")
axes[0].set_title("Ground Truth", fontweight="bold")
axes[1].imshow(best1, cmap="gray")
axes[1].set_title(f"Restored Data1 (PSNR: {m1["psnr_db"]:.1f} dB)", fontweight="bold")
axes[2].imshow(best2, cmap="gray")
axes[2].set_title(f"Restored Data2 (PSNR: {m2["psnr_db"]:.1f} dB)", fontweight="bold")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Data1 -> PSNR: {m1["psnr_db"]:.2f} dB, SSIM: {m1["ssim"]:.4f}, Relative L2: {m1["relative_l2"]:.4f}")
print(f"Data2 -> PSNR: {m2["psnr_db"]:.2f} dB, SSIM: {m2["ssim"]:.4f}, Relative L2: {m2["relative_l2"]:.4f}")